<a href="https://colab.research.google.com/github/Karthikreddy1010/Electric-poles-and-wires-detection/blob/main/OOB_Correctiuons.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!unzip /content/OBB-CV.zip

In [ ]:
!unzip /content/dataset_qa_v4.zip


In [ ]:
"""
colab_bootstrap.py
===================
PASTE THIS ENTIRE FILE'S CONTENT INTO THE FIRST CELL of your notebook
(or `%run` it if you uploaded it), BEFORE importing anything from
dataset_qa. It does not assume any particular unzip location - it
searches common Colab roots for wherever you extracted the package and
fixes sys.path automatically, then verifies the fix actually works.

Why this is needed: `pipeline.py` depends on its sibling files
(geometry.py, visibility.py, mask_refinement.py, etc.) sitting next to
it on sys.path. If you `%run pipeline.py` or paste pipeline.py's
content into a cell with nothing else set up, Python has no way to
find those siblings - this script finds them for you.

After running this cell, use EITHER:

    from dataset_qa.config import Config
    from dataset_qa.pipeline import run_pipeline
    from dataset_qa import reports, cleaning

  OR, if you specifically want the bare module names (e.g. you're
  about to paste pipeline.py's own content into the next cell):

    import pipeline, geometry, config   # etc.
"""

import glob
import os
import sys

_SEARCH_ROOTS = ["/content", "/content/drive", os.getcwd(), "/kaggle/working", "/home", os.path.expanduser("~")]


def _find_dataset_qa_dirs():
    """Returns (outer_dir, inner_dir):
       outer_dir = folder containing main.py + config.yaml + the inner dataset_qa/ package
       inner_dir = folder containing pipeline.py + its siblings (geometry.py, etc.)
    Both may be the same folder in unusual layouts; either may be None
    if not found."""
    for root in _SEARCH_ROOTS:
        if not root or not os.path.isdir(root):
            continue
        # look for pipeline.py anywhere under this root (bounded depth via glob's **)
        for pipeline_path in glob.glob(os.path.join(root, "**", "pipeline.py"), recursive=True):
            inner = os.path.dirname(pipeline_path)
            if not os.path.exists(os.path.join(inner, "geometry.py")):
                continue  # some other unrelated pipeline.py
            outer = os.path.dirname(inner)
            # sanity check: does outer look like the package root? (has main.py or config.yaml)
            if not (os.path.exists(os.path.join(outer, "main.py")) or os.path.exists(os.path.join(outer, "config.yaml"))):
                outer = None
            return outer, inner
    return None, None


def bootstrap(verbose: bool = True):
    outer, inner = _find_dataset_qa_dirs()

    if inner is None:
        raise RuntimeError(
            "Could not locate dataset_qa's pipeline.py (with its sibling geometry.py) under "
            f"any of: {_SEARCH_ROOTS}. Make sure you've unzipped dataset_qa_v4.zip somewhere "
            "under /content first, e.g.:\n"
            "    !unzip -q /content/dataset_qa_v4.zip -d /content/\n"
            "then re-run this bootstrap cell."
        )

    for p in (outer, inner):
        if p and p not in sys.path:
            sys.path.insert(0, p)

    if verbose:
        print(f"[dataset_qa bootstrap] inner (siblings, e.g. geometry.py) -> {inner}")
        print(f"[dataset_qa bootstrap] outer (package root, e.g. main.py) -> {outer}")

    # Verify both import styles actually work now, so failures surface
    # here with a clear message instead of downstream.
    problems = []
    try:
        import importlib
        if outer:
            importlib.import_module("dataset_qa.pipeline")
            if verbose:
                print("[dataset_qa bootstrap] OK: `from dataset_qa import pipeline` works")
    except Exception as e:  # noqa: BLE001
        problems.append(f"package-style import failed: {e}")

    try:
        import importlib
        importlib.import_module("geometry")
        importlib.import_module("pipeline")
        if verbose:
            print("[dataset_qa bootstrap] OK: bare `import pipeline` / `import geometry` works")
    except Exception as e:  # noqa: BLE001
        problems.append(f"standalone-style import failed: {e}")

    if problems and verbose:
        print("[dataset_qa bootstrap] WARNING - one or more import styles still failed:")
        for p in problems:
            print("   -", p)
        print("At least one style should have worked; if BOTH failed, check the paths printed "
              "above look right, and that the .zip extracted its full folder structure (not just "
              "pipeline.py alone).")

    return outer, inner


if __name__ == "__main__":
    bootstrap()

[dataset_qa bootstrap] inner (siblings, e.g. geometry.py) -> /content/dataset_qa/dataset_qa
[dataset_qa bootstrap] outer (package root, e.g. main.py) -> /content/dataset_qa
[dataset_qa bootstrap] OK: `from dataset_qa import pipeline` works
[dataset_qa bootstrap] OK: bare `import pipeline` / `import geometry` works


In [ ]:
from dataset_qa.config import Config
from dataset_qa.pipeline import run_pipeline
from dataset_qa import reports, cleaning

cfg = Config.from_yaml("/content/OBB-CV/data.yaml")   # see note below
cfg.paths.dataset_root = "/content/OBB-CV"             # parent of train/val/test, NOT .../train
cfg.paths.splits = ["train", "val", "test"]            # your folders are "val", not "valid"
cfg.paths.output_root = "/content/qa_output"

df, issues, records = run_pipeline(cfg)
reports.write_all_reports(df, issues, cfg.paths.splits, cfg.paths.output_root, cfg=cfg)
cleaning.clean_dataset(records, cfg.paths.dataset_root, cfg.paths.output_root)

Pass 2: 100%|██████████| 7476/7476 [00:00<00:00, 931762.89it/s]


{'keep_images': 6792,
 'keep_labels': 8245,
 'corrected_labels': 0,
 'review_images': 346,
 'review_labels': 350,
 'removed_images': 0,
 'removed_labels': 0,
 'export_validation_dropped': 0,
 'images_skipped_zero_kept_labels': 236}

In [ ]:
!zip -r /content/qa_output/corrected_dataset.zip /content/qa_output/corrected_dataset